# Human Activity Recognition using Hidden Markov Models
### Accelerometer + Gyroscope — Pixel 3a (Android) · Sensor Logger App
---
**Activities :** Still · Standing · Walking · Jumping  
**Sensors    :** Accelerometer (x, y, z) + Gyroscope (x, y, z) — merged CSV  
**Pipeline   :** Load → Resample → Window → Feature Extraction → HMM (Baum–Welch + Viterbi) → Evaluate

## 0 · Imports & Global Configuration

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ML-Techniques-II-formative-2

Mounted at /content/drive
/content/drive/MyDrive/ML-Techniques-II-formative-2


In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from itertools import permutations
from pathlib import Path
import warnings, os
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi'        : 130,
    'font.size'         : 11,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

# ─── PATHS ────────────────────────────────────────────────────────────────────
TRAIN_DIR = Path('dataset/train')
TEST_DIR  = Path('dataset/test')
OUT_DIR   = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

# ─── SAMPLING & WINDOWING ─────────────────────────────────────────────────────
TARGET_FS    = 50          # Hz  — harmonised rate for both devices
WINDOW_SEC   = 2.0         # seconds per window
OVERLAP      = 0.5         # 50 % overlap between consecutive windows
WINDOW_SAMP  = int(TARGET_FS * WINDOW_SEC)        # 100 samples
STEP_SAMP    = int(WINDOW_SAMP * (1 - OVERLAP))   #  50 samples

# ─── ACTIVITIES ───────────────────────────────────────────────────────────────
ACTIVITIES = ['still', 'standing', 'walking', 'jumping']
ACT2ID     = {a: i for i, a in enumerate(ACTIVITIES)}
ID2ACT     = {i: a for i, a in enumerate(ACTIVITIES)}
COLORS     = {'still':'#4e79a7','standing':'#f28e2b',
              'walking':'#59a14f','jumping':'#e15759'}

FEATURE_NAMES = [
    'RMS Accel',   'Var Accel',    'SMA',
    'Mean Gyro',   'Var Gyro',     'Corr ax-ay',
    'Dom Freq',    'Spec Energy',  'Spec Entropy',
]

print(f'Window : {WINDOW_SAMP} samples = {WINDOW_SEC} s  @  {TARGET_FS} Hz')
print(f'Step   : {STEP_SAMP} samples ({int(OVERLAP*100)}% overlap)')
print(f'Activities: {ACTIVITIES}')

Window : 100 samples = 2.0 s  @  50 Hz
Step   : 50 samples (50% overlap)
Activities: ['still', 'standing', 'walking', 'jumping']


## 1 · Data Collection Overview

Data were recorded with **Sensor Logger** and exported as merged CSVs  
(one file per recording, columns: `time · seconds_elapsed · accel_z/y/x · gyro_x/y/z`).

| Participant | Device | Native Sample Rate | Role |
|---|---|---|---|
| Member 1 | Pixel 3a (Android) | ~50 Hz | Training + Test |
| Member 2 | iPhone | ~50 Hz | Training + Test |

### File Naming Convention
```
dataset/train/
  still_1.csv  …  still_10.csv
  standing_1.csv  …  standing_10.csv
  walking_1.csv   …  walking_10.csv
  jumping_1.csv   …  jumping_10.csv

dataset/test/
  still_1.csv    # unseen test files
  standing_1.csv
  walking_1.csv
  jumping_1.csv
```

### Window Size Justification
At **50 Hz**, 1 sample = 20 ms.  
A **2-second window (100 samples)** was chosen because:
- It spans ≥ 1 full walking gait cycle (~0.9–1.2 s) and ≥ 1 full jump cycle.
- FFT resolution = 1/2 s = **0.5 Hz** — sufficient to separate still/standing  
  (near-DC) from walking (~1–2 Hz) and jumping (~2–3 Hz).
- Short enough to capture activity transitions for the HMM temporal model.

**50% overlap** doubles the number of windows and smooths feature transitions.

### Sampling Rate Harmonisation
Both devices export `seconds_elapsed` timestamps. Every recording is  
**linearly resampled to exactly 50 Hz** before any processing, ensuring  
all windows and frequency features are directly comparable.

## 2 · Data Loading & Resampling

In [21]:
# ─── Expected merged column aliases (handles minor naming variants) ────────────
COL_MAP = {
    # Accelerometer
    'accel_x':'ax', 'accel_y':'ay', 'accel_z':'az',
    'acceleration_x':'ax', 'acceleration_y':'ay', 'acceleration_z':'az',
    'x':'ax', 'y':'ay', 'z':'az',           # fallback for old single-sensor files
    # Gyroscope
    'gyro_x':'gx', 'gyro_y':'gy', 'gyro_z':'gz',
    'gyroscope_x':'gx', 'gyroscope_y':'gy', 'gyroscope_z':'gz',
}


def load_merged(path: Path) -> pd.DataFrame:
    """
    Load one merged CSV (accel + gyro in the same file),
    normalise column names, and resample to TARGET_FS via
    linear interpolation over seconds_elapsed.

    Expected columns (any capitalisation / separator):
        time  seconds_elapsed  accel_z  accel_y  accel_x  gyro_x  gyro_y  gyro_z
    """
    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

    # Rename to canonical names
    df = df.rename(columns=COL_MAP)

    required = ['seconds_elapsed', 'ax', 'ay', 'az', 'gx', 'gy', 'gz']
    missing  = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{path.name}: missing columns {missing}. '
                         f'Found: {list(df.columns)}')

    df = df.sort_values('seconds_elapsed').dropna(subset=required).reset_index(drop=True)

    # ── Resample to TARGET_FS ──────────────────────────────────────────────────
    t_orig = df['seconds_elapsed'].values
    t_new  = np.arange(t_orig[0], t_orig[-1], 1.0 / TARGET_FS)

    resampled = {'seconds_elapsed': t_new}
    for col in ['ax', 'ay', 'az', 'gx', 'gy', 'gz']:
        resampled[col] = interp1d(
            t_orig, df[col].values, kind='linear',
            bounds_error=False, fill_value='extrapolate'
        )(t_new)

    return pd.DataFrame(resampled)


def discover_recordings(data_dir: Path):
    """
    Scan data_dir for CSVs following '{activity}_{n}.csv' pattern.
    Returns:
        files : {activity: [Path, ...]}
    """
    files = {a: [] for a in ACTIVITIES}

    for p in sorted(data_dir.glob('*.csv')):
        stem = p.stem.lower()
        # Remove common prefixes
        stem_clean = stem.removeprefix('test_').removeprefix('train_')
        act = next((a for a in ACTIVITIES if stem_clean.startswith(a)), None)
        if act is None:
            print(f'  [skip] {p.name}  — no matching activity in name')
            continue
        files[act].append(p)

    return files


# ── Load everything ────────────────────────────────────────────────────────────
train_files = discover_recordings(TRAIN_DIR)
test_files  = discover_recordings(TEST_DIR)

def load_group(file_dict):
    data = {a: [] for a in ACTIVITIES}
    for act, paths in file_dict.items():
        for p in paths:
            try:
                df = load_merged(p)
                dur = df['seconds_elapsed'].iloc[-1] - df['seconds_elapsed'].iloc[0]
                data[act].append(df)
                print(f'  ✓ {p.name:35s} → {len(df):4d} samples  ({dur:.2f} s)')
            except Exception as e:
                print(f'  ✗ {p.name}: {e}')
    return data

print('── Training files ───────────────────────────────────────')
train_data = load_group(train_files)

print('\n── Test files ───────────────────────────────────────────')
test_data  = load_group(test_files)

print('\n── Summary ──────────────────────────────────────────────')
for act in ACTIVITIES:
    n   = len(train_data[act])
    dur = sum((d['seconds_elapsed'].iloc[-1]-d['seconds_elapsed'].iloc[0])
              for d in train_data[act])
    nt  = len(test_data[act])
    print(f'  {act:10s}: {n:2d} train files  (~{dur:.0f} s)   |   {nt} test files')

  [skip] eliel_Jumping_eliel_01.csv.csv  — no matching activity in name
── Training files ───────────────────────────────────────
  ✓ still_eliel_01.csv.csv              →  500 samples  (9.98 s)
  ✓ still_eliel_02.csv.csv              →  479 samples  (9.56 s)
  ✓ still_eliel_03.csv.csv              →  492 samples  (9.82 s)
  ✓ still_eliel_04.csv.csv              →  542 samples  (10.82 s)
  ✓ still_eliel_05.csv.csv              →  513 samples  (10.24 s)
  ✓ still_eliel_06.csv.csv              →  494 samples  (9.86 s)
  ✓ still_merged_1.csv                  →  529 samples  (10.56 s)
  ✓ still_merged_2.csv                  →  530 samples  (10.58 s)
  ✓ still_merged_3.csv                  →  531 samples  (10.60 s)
  ✓ still_merged_4.csv                  →  524 samples  (10.46 s)
  ✓ still_merged_5.csv                  →  527 samples  (10.52 s)
  ✓ still_merged_6.csv                  →  527 samples  (10.52 s)
  ✓ standing_eliel_01.csv.csv           →  501 samples  (10.00 s)
  ✓ standing_eli

## 3 · Visualisation of Raw Sensor Signals

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
fig.suptitle('Raw Sensor Signals — One Representative Recording per Activity',
             fontsize=14, fontweight='bold')

for row, act in enumerate(ACTIVITIES):
    samples = train_data[act] or test_data[act]
    if not samples:
        for col in range(2):
            axes[row, col].set_visible(False)
        continue
    df = samples[0]
    t  = df['seconds_elapsed'].values - df['seconds_elapsed'].values[0]
    c  = COLORS[act]

    ax_a = axes[row, 0]
    for axis_col, lbl in zip(['ax','ay','az'], ['x','y','z']):
        ax_a.plot(t, df[axis_col], alpha=0.85, lw=1, label=lbl)
    ax_a.set_title(f'{act.capitalize()} — Accelerometer (m/s²)', color=c, fontsize=11)
    ax_a.set_ylabel('m/s²')
    ax_a.legend(loc='upper right', fontsize=8, ncol=3)

    ax_g = axes[row, 1]
    for axis_col, lbl in zip(['gx','gy','gz'], ['x','y','z']):
        ax_g.plot(t, df[axis_col], alpha=0.85, lw=1, label=lbl)
    ax_g.set_title(f'{act.capitalize()} — Gyroscope (rad/s)', color=c, fontsize=11)
    ax_g.set_ylabel('rad/s')
    ax_g.legend(loc='upper right', fontsize=8, ncol=3)

for ax in axes[-1]:
    ax.set_xlabel('Time (s)')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_raw_signals.png', bbox_inches='tight')
plt.show()

## 4 · Frequency Analysis — FFT per Activity

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
fig.suptitle('Average FFT Magnitude of Acceleration — by Activity', fontweight='bold')

for ax, act in zip(axes, ACTIVITIES):
    samples = train_data[act] or test_data[act]
    if not samples:
        ax.set_visible(False); continue
    spectra = []
    for df in samples:
        a_mag = np.sqrt(df['ax']**2 + df['ay']**2 + df['az']**2).values
        n      = len(a_mag)
        freqs  = np.fft.rfftfreq(n, d=1.0/TARGET_FS)
        mag    = np.abs(np.fft.rfft(a_mag * np.hanning(n)))
        spectra.append(mag / (mag.max() + 1e-9))   # normalise per recording
    # Interpolate all to same freq grid then average
    max_len = max(len(s) for s in spectra)
    grid = np.fft.rfftfreq(max_len*2, d=1.0/TARGET_FS)
    grid = grid[grid <= 15]
    mean_spec = np.mean(
        [np.interp(grid, np.fft.rfftfreq(len(s)*2, d=1.0/TARGET_FS)[:len(s)], s)
         for s in spectra], axis=0)
    ax.fill_between(grid, mean_spec, alpha=0.4, color=COLORS[act])
    ax.plot(grid, mean_spec, color=COLORS[act], lw=2)
    ax.set_title(act.capitalize(), color=COLORS[act], fontweight='bold')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Normalised Magnitude')
    ax.axvspan(0.5, 5, alpha=0.07, color='grey', label='Motion band')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2_fft_per_activity.png', bbox_inches='tight')
plt.show()

## 5 · Feature Extraction

### Feature Justification

| # | Feature | Domain | Why it helps |
|---|---------|--------|-------------|
| 1 | **RMS of accel magnitude** | Time | Overall movement intensity; jumping >> walking >> standing ≈ still |
| 2 | **Variance of accel magnitude** | Time | Separates still (≈ 0) from all active states |
| 3 | **Signal Magnitude Area (SMA)** | Time | Orientation-invariant activity intensity; complements RMS |
| 4 | **Mean gyro magnitude** | Time | Rotational activity; near-zero when still/standing |
| 5 | **Variance of gyro magnitude** | Time | Distinguishes steady rotation (walking) from explosive rotation (jumping) |
| 6 | **Cross-correlation ax–ay** | Time | Captures coordinated arm/leg swing patterns unique to walking |
| 7 | **Dominant frequency (accel)** | Frequency | Encodes step cadence (~1–2 Hz walking) and jump rate (~2–3 Hz) |
| 8 | **Spectral energy (0.5–5 Hz)** | Frequency | Human motion band; static activities have near-zero band energy |
| 9 | **Spectral entropy (accel)** | Frequency | Walking is highly periodic (low entropy); jumping less so; still highest entropy |

### Normalisation
All features are **Z-score normalised** (subtract mean, divide by std — `StandardScaler`).  
Justification: features operate on very different scales (Hz vs m/s² vs dimensionless).  
Z-score ensures each feature contributes equally to the Gaussian emission likelihood.

In [ ]:
def spectral_features(a_mag: np.ndarray, fs: float = TARGET_FS):
    """Return (dominant_freq, spectral_energy_0.5-5Hz, spectral_entropy)."""
    n      = len(a_mag)
    freqs  = np.fft.rfftfreq(n, d=1.0/fs)
    mag    = np.abs(np.fft.rfft(a_mag * np.hanning(n)))
    power  = mag ** 2
    total  = power.sum() + 1e-12

    # Dominant frequency (ignore DC)
    nz_idx  = np.where(freqs > 0.1)[0]
    dom_f   = freqs[nz_idx[np.argmax(power[nz_idx])]]

    # Spectral energy in human-motion band 0.5–5 Hz
    band    = (freqs >= 0.5) & (freqs <= 5.0)
    spec_e  = power[band].sum() / total

    # Spectral entropy (normalised)
    p       = power / total
    p       = p[p > 0]
    entropy = -np.sum(p * np.log2(p + 1e-12)) / np.log2(len(p) + 1)

    return dom_f, spec_e, entropy


def extract_windows(df: pd.DataFrame) -> np.ndarray:
    """
    Slide WINDOW_SAMP window with STEP_SAMP step over df.
    Returns feature matrix of shape (n_windows, 9).
    """
    ax_v  = df['ax'].values;  ay_v = df['ay'].values;  az_v = df['az'].values
    gx_v  = df['gx'].values;  gy_v = df['gy'].values;  gz_v = df['gz'].values
    a_mag = np.sqrt(ax_v**2 + ay_v**2 + az_v**2)
    g_mag = np.sqrt(gx_v**2 + gy_v**2 + gz_v**2)

    rows, n = [], len(df)
    start = 0
    while start + WINDOW_SAMP <= n:
        sl   = slice(start, start + WINDOW_SAMP)
        aw   = a_mag[sl];  gw = g_mag[sl]
        axw  = ax_v[sl];   ayw = ay_v[sl]

        rms_acc  = np.sqrt(np.mean(aw**2))
        var_acc  = np.var(aw)
        sma      = (np.abs(ax_v[sl]).sum() +
                    np.abs(ay_v[sl]).sum() +
                    np.abs(az_v[sl]).sum()) / WINDOW_SAMP
        mean_gyr = np.mean(gw)
        var_gyr  = np.var(gw)
        cc       = np.corrcoef(axw, ayw)[0, 1]
        if np.isnan(cc): cc = 0.0
        dom_f, spec_e, entropy = spectral_features(aw)

        rows.append([rms_acc, var_acc, sma, mean_gyr, var_gyr,
                     cc, dom_f, spec_e, entropy])
        start += STEP_SAMP

    return np.array(rows) if rows else np.empty((0, 9))


def build_feature_set(data_dict):
    """Apply extract_windows to every recording, return list of arrays + label array."""
    seqs, labels = [], []
    for act in ACTIVITIES:
        lbl = ACT2ID[act]
        for df in data_dict[act]:
            F = extract_windows(df)
            if F.shape[0] > 0:
                seqs.append(F)
                labels.extend([lbl] * len(F))
    return seqs, np.array(labels)


train_seqs, y_train_raw = build_feature_set(train_data)
test_seqs,  y_test_raw  = build_feature_set(test_data)

print('Feature extraction complete')
for act in ACTIVITIES:
    lbl   = ACT2ID[act]
    n_tr  = (y_train_raw == lbl).sum()
    n_te  = (y_test_raw  == lbl).sum() if len(y_test_raw) else 0
    print(f'  {act:10s}: {n_tr:3d} train windows   {n_te:3d} test windows')
print(f'  {"TOTAL":10s}: {len(y_train_raw):3d} train           {len(y_test_raw):3d} test')

## 6 · Normalisation & Feature Analysis

In [ ]:
# Fit scaler on TRAINING data only, then apply to both sets
X_train_raw = np.vstack(train_seqs) if train_seqs else np.empty((0, 9))
X_test_raw  = np.vstack(test_seqs)  if test_seqs  else np.empty((0, 9))

scaler    = StandardScaler()
X_train_n = scaler.fit_transform(X_train_raw)
X_test_n  = scaler.transform(X_test_raw) if X_test_raw.shape[0] else X_test_raw

# Per-activity normalised sequences (keep sequence boundaries for HMM)
train_seqs_n = [scaler.transform(s) for s in train_seqs]

print('Scaler fitted on training data only  (mean=0, std=1)')
print(f'  X_train : {X_train_n.shape}   X_test : {X_test_n.shape}')

In [ ]:
# ── Box-plot: feature distributions per activity ───────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('Feature Distributions per Activity (training data, normalised)',
             fontsize=13, fontweight='bold')

for idx, (ax, fname) in enumerate(zip(axes.flat, FEATURE_NAMES)):
    data_bp, labels_bp = [], []
    for act in ACTIVITIES:
        lbl  = ACT2ID[act]
        vals = X_train_n[y_train_raw == lbl, idx]
        if len(vals):
            data_bp.append(vals)
            labels_bp.append(act.capitalize())
    if not data_bp: continue
    bp = ax.boxplot(data_bp, patch_artist=True, notch=False, labels=labels_bp)
    for patch, act in zip(bp['boxes'], [a for a in ACTIVITIES if (y_train_raw==ACT2ID[a]).sum()]):
        patch.set_facecolor(COLORS[act]); patch.set_alpha(0.75)
    ax.set_title(fname, fontsize=10)
    ax.tick_params(axis='x', labelsize=9)
    ax.axhline(0, color='grey', lw=0.8, ls='--')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig3_feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature correlation heatmap ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = pd.DataFrame(X_train_n, columns=FEATURE_NAMES).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, annot_kws={'size': 8}, ax=ax)
ax.set_title('Feature Correlation Matrix (training set, normalised)', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig4_feature_correlation.png', bbox_inches='tight')
plt.show()

## 7 · Gaussian HMM — Full Implementation

The model comprises:
- **Hidden states** Z ∈ {Still, Standing, Walking, Jumping}
- **Observations** X = 9-dimensional normalised feature vectors
- **Transition matrix A** — probability of moving between activities
- **Emission B** — diagonal-covariance Gaussian per state
- **Initial distribution π**

Training uses **Baum–Welch (EM)** with log-likelihood convergence criterion.  
Decoding uses the **Viterbi algorithm** (log-space, backtracking).

In [ ]:
class GaussianHMM:
    """
    Diagonal-covariance Gaussian HMM implemented from scratch with NumPy.

    Parameters
    ----------
    n_states : int    Number of hidden states (= number of activities)
    n_iter   : int    Maximum Baum–Welch iterations
    tol      : float  Convergence threshold on log-likelihood change
    verbose  : bool
    """

    def __init__(self, n_states: int = 4, n_iter: int = 300,
                 tol: float = 1e-4, verbose: bool = True):
        self.n_states = n_states
        self.n_iter   = n_iter
        self.tol      = tol
        self.verbose  = verbose

    # ── Parameter initialisation ──────────────────────────────────────────────
    def _init_params(self, X_all: np.ndarray, state_labels=None):
        K, D = self.n_states, X_all.shape[1]

        # Transition: strong self-loop bias (activities persist)
        A = np.full((K, K), (1 - 0.9) / (K - 1))
        np.fill_diagonal(A, 0.9)
        A /= A.sum(axis=1, keepdims=True)

        # Uniform initial distribution
        pi = np.ones(K) / K

        # Emissions: initialise from per-class statistics when labels available
        if state_labels is not None and len(np.unique(state_labels)) == K:
            mu  = np.array([
                X_all[state_labels == k].mean(axis=0)
                if (state_labels == k).sum() > 0 else X_all.mean(axis=0)
                for k in range(K)
            ])
            cov = np.array([
                np.var(X_all[state_labels == k], axis=0)
                if (state_labels == k).sum() > 1 else np.ones(D)
                for k in range(K)
            ])
        else:
            # k-means style split along first principal component
            idx   = np.argsort(X_all[:, 0])
            chunk = max(len(X_all) // K, 1)
            mu    = np.array([X_all[idx[k*chunk:(k+1)*chunk]].mean(axis=0)
                              for k in range(K)])
            cov   = np.array([X_all[idx[k*chunk:(k+1)*chunk]].var(axis=0)
                              for k in range(K)])

        self.pi_  = pi
        self.A_   = A
        self.mu_  = mu
        self.cov_ = np.clip(cov, 1e-3, None)

    # ── Log-emission probability  B(t, k) ─────────────────────────────────────
    def _log_emission(self, X: np.ndarray) -> np.ndarray:
        """Return (T × K) matrix of log p(x_t | z_t = k)."""
        T, D   = X.shape
        log_b  = np.zeros((T, self.n_states))
        for k in range(self.n_states):
            cov_k   = np.maximum(self.cov_[k], 1e-6)
            log_det = np.sum(np.log(cov_k))
            diff    = X - self.mu_[k]
            maha    = np.sum(diff**2 / cov_k, axis=1)
            log_b[:, k] = -0.5 * (D * np.log(2 * np.pi) + log_det + maha)
        return log_b

    # ── Forward pass (log-space) ──────────────────────────────────────────────
    def _forward(self, log_b: np.ndarray):
        T, K     = log_b.shape
        log_A    = np.log(self.A_ + 1e-300)
        log_alpha = np.full((T, K), -np.inf)
        log_alpha[0] = np.log(self.pi_ + 1e-300) + log_b[0]
        for t in range(1, T):
            for k in range(K):
                log_alpha[t, k] = (
                    np.logaddexp.reduce(log_alpha[t-1] + log_A[:, k])
                    + log_b[t, k]
                )
        ll = np.logaddexp.reduce(log_alpha[-1])
        return log_alpha, ll

    # ── Backward pass (log-space) ─────────────────────────────────────────────
    def _backward(self, log_b: np.ndarray) -> np.ndarray:
        T, K    = log_b.shape
        log_A   = np.log(self.A_ + 1e-300)
        log_beta = np.full((T, K), -np.inf)
        log_beta[-1] = 0.0
        for t in range(T - 2, -1, -1):
            for i in range(K):
                log_beta[t, i] = np.logaddexp.reduce(
                    log_A[i] + log_b[t+1] + log_beta[t+1]
                )
        return log_beta

    # ── Baum–Welch (EM) ───────────────────────────────────────────────────────
    def fit(self, sequences: list, state_labels=None):
        """
        sequences   : list of 2-D arrays (T_i × D), one per recording.
        state_labels: optional flat label array (stacked sequences) for
                      informed emission initialisation.
        """
        X_all = np.vstack(sequences)
        self._init_params(X_all, state_labels)
        K = self.n_states

        prev_ll          = -np.inf
        self.ll_history_ = []

        for iteration in range(self.n_iter):

            # ── E-step ────────────────────────────────────────────────────────
            gamma_list = []
            xi_acc     = np.zeros((K, K))
            total_ll   = 0.0

            for seq in sequences:
                if len(seq) < 2:
                    continue
                log_b          = self._log_emission(seq)
                log_alpha, ll  = self._forward(log_b)
                log_beta       = self._backward(log_b)
                total_ll      += ll

                # γ
                log_gamma  = log_alpha + log_beta
                log_gamma -= np.logaddexp.reduce(log_gamma, axis=1, keepdims=True)
                gamma_list.append(np.exp(log_gamma))

                # ξ — summed over time
                log_A = np.log(self.A_ + 1e-300)
                T_seq = len(seq)
                for t in range(T_seq - 1):
                    log_xi  = (log_alpha[t, :, None] + log_A
                               + log_b[t+1, None, :] + log_beta[t+1, None, :])
                    log_xi -= np.logaddexp.reduce(log_xi.ravel())
                    xi_acc += np.exp(log_xi)

            self.ll_history_.append(total_ll)
            if self.verbose and (iteration % 25 == 0 or iteration < 4):
                print(f'  iter {iteration:4d}   log-likelihood = {total_ll:.4f}')

            # ── Convergence check ─────────────────────────────────────────────
            delta_ll = abs(total_ll - prev_ll)
            if delta_ll < self.tol:
                if self.verbose:
                    print(f'   Converged at iteration {iteration}')
                    print(f'    ΔLL = {delta_ll:.2e}  <  ε = {self.tol}')
                break
            prev_ll = total_ll

            # ── M-step ────────────────────────────────────────────────────────
            # π
            self.pi_ = np.clip(
                np.mean([g[0] for g in gamma_list], axis=0), 1e-10, None)
            self.pi_ /= self.pi_.sum()

            # A
            row_sum  = xi_acc.sum(axis=1, keepdims=True) + 1e-300
            self.A_  = np.clip(xi_acc / row_sum, 1e-10, None)
            self.A_ /= self.A_.sum(axis=1, keepdims=True)

            # μ, σ²
            gamma_all = np.vstack(gamma_list)   # (T_total × K)
            for k in range(K):
                g_k   = gamma_all[:, k:k+1]     # (T, 1)
                denom = g_k.sum() + 1e-300
                self.mu_[k]  = (g_k * X_all).sum(axis=0) / denom
                diff         = X_all - self.mu_[k]
                self.cov_[k] = np.clip(
                    (g_k * diff**2).sum(axis=0) / denom, 1e-4, None)

        return self

    # ── Viterbi decoding ──────────────────────────────────────────────────────
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Viterbi algorithm: returns the most likely hidden state sequence.
        Uses log-space DP with backtracking pointers.
        """
        T, K   = len(X), self.n_states
        log_b  = self._log_emission(X)
        log_A  = np.log(self.A_ + 1e-300)

        delta  = np.full((T, K), -np.inf)
        psi    = np.zeros((T, K), dtype=int)
        delta[0] = np.log(self.pi_ + 1e-300) + log_b[0]

        for t in range(1, T):
            scores   = delta[t-1, :, None] + log_A   # (K, K)
            psi[t]   = np.argmax(scores, axis=0)
            delta[t] = np.max(scores, axis=0) + log_b[t]

        # Backtrack
        path      = np.zeros(T, dtype=int)
        path[-1]  = np.argmax(delta[-1])
        for t in range(T - 2, -1, -1):
            path[t] = psi[t+1, path[t+1]]
        return path

    def score(self, X: np.ndarray) -> float:
        """Total log-likelihood of sequence X under the current model."""
        _, ll = self._forward(self._log_emission(X))
        return ll


print('GaussianHMM class ready ✓')

## 8 · Training — Baum–Welch

In [ ]:
# Build per-recording normalised sequences with matching flat label array
labels_train_flat = []
for act in ACTIVITIES:
    lbl = ACT2ID[act]
    for df in train_data[act]:
        F = extract_windows(df)
        labels_train_flat.extend([lbl] * len(F))
y_train_labels = np.array(labels_train_flat)

print(f'Training HMM on {len(train_seqs_n)} sequences '
      f'({len(X_train_n)} total windows)...\n')

hmm = GaussianHMM(n_states=4, n_iter=300, tol=1e-4, verbose=True)
hmm.fit(train_seqs_n, state_labels=y_train_labels)

print(f'\nBaum–Welch complete  ({len(hmm.ll_history_)} iterations)')
print(f'Final log-likelihood : {hmm.ll_history_[-1]:.4f}')

## 9 · State–Activity Mapping
Baum–Welch does not guarantee label alignment. We find the permutation of
HMM states that maximises training accuracy.